# 📗 텍스트 벡터화와 임베딩 — 임베딩 기초

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간까지 우리는 텍스트를 전처리하고 **단어의 개수**를 세어 숫자로 바꿨습니다(BoW·TF-IDF). 하지만 그 방식은 **단어가 겹쳐야만** 비슷하다고 판단합니다 — "영화가 재미있다"와 "작품이 훌륭하다"는 뜻이 비슷한데도 겹치는 단어가 없어 "전혀 다른 문장"으로 취급됐죠. 이번 시간엔 문장의 **의미 자체를 좌표(벡터)로** 바꾸는 **임베딩(embedding)** 을 배웁니다. 임베딩을 쓰면 단어가 겹치지 않아도 **뜻이 가까운 문장**을 찾을 수 있습니다 — 코사인 유사도, 의미 검색, 임베딩 시각화까지 차례로 익힙니다.

## ⏪ 복습 — 지난 시간: 텍스트 전처리와 TF-IDF

지난 시간에 배운 것이 이번 시간의 **출발점이자 비교 대상**입니다.

- **텍스트 전처리** — 문장을 토큰(단어)으로 나누고, 불용어를 걸러 정리했습니다.
- **BoW · TF-IDF** — 문서를 "어떤 단어가 몇 번 나왔나"의 표로 바꿨습니다. 단어마다 칸이 하나씩 있어 **대부분이 0인 아주 긴 벡터**(희소 벡터)가 나왔죠.
- **한계** — 이 방식은 **같은 단어**가 나와야 비슷하다고 봅니다. 뜻은 같지만 다른 단어로 쓴 문장("긴장감 넘친다" vs "손에 땀을 쥔다")은 겹치는 단어가 없어 **안 비슷하다**고 나옵니다.

> 오늘의 한 문장: **"단어를 세는 것에서 의미를 좌표로 옮기는 것으로."**

**오늘의 목표**

- [ ] **희소 벡터(TF-IDF)와 밀집 벡터(임베딩)** 의 차이를 설명한다.
- [ ] HuggingFace **`SentenceTransformer`** 로 문장을 **768차원 벡터**로 바꾸고, **BERT→풀링→SBERT** 로 그 벡터가 만들어지는 구조와 차원의 의미를 설명한다.
- [ ] **벡터의 내적·길이(노름)** 를 수식으로 이해하고, **코사인 유사도 공식**을 손으로 계산한다.
- [ ] **코사인 유사도**로 두 문장이 얼마나 비슷한지 −1~1 로 잰다.
- [ ] **유클리드 거리·내적**과 코사인을 비교하고, **L2 정규화**하면 셋이 같아지는 것을 확인한다.
- [ ] **의미 검색** — 질문을 임베딩해 가장 가까운 문서 top-k 를 찾는다.
- [ ] **OpenAI Embeddings API** 로 같은 일(문장→벡터)을 해 본다(키가 있으면 실제 호출).
- [ ] **HuggingFace 허브에서 임베딩 모델을 찾고**, 차원·최대 입력 길이·라이선스를 보고 고른다.
- [ ] **UMAP** 으로 768차원 임베딩을 2차원 그림으로 펼쳐 본다.

아래 두 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from umap import UMAP

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

# 그래프에 일관되게 쓸 장르별 색
GENRE_COLORS = {'액션': '#3B82F6', '로맨스': '#F43F5E', '코미디': '#F59E0B'}

In [ ]:
# [제공 코드] 한국어 문장 임베딩 모델을 불러옵니다(문장 -> 768차원 벡터).
# 처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

## 데이터 살펴보기 — 영화 리뷰 15건

이번 시간엔 짧은 **영화 리뷰** 데이터를 씁니다. 세 장르(**액션·로맨스·코미디**)에서 각각 5건씩, 모두 15건입니다. 각 리뷰가 어느 장르인지(`genre`) 붙어 있어, 임베딩이 **장르(뜻)의 차이를 잘 담는지** 눈으로 확인하기 좋습니다.

새 데이터를 만나면 분석 전에 **생김새부터** 봅니다 — 앞부분(`head`)·구조와 결측(`info`)·장르 분포(`value_counts`).

In [ ]:
df = pd.read_csv('data/movie_reviews.csv')

print('데이터 크기:', df.shape)
print('\n[앞부분] head()')
display(df.head())
print('\n[구조와 결측] info()')
df.info()
print('\n[장르(genre) 분포]')
display(df['genre'].value_counts().to_frame('건수'))

---
# 1. 왜 임베딩인가 — 희소 벡터 vs 밀집 벡터

## 왜 필요할까요?
지난 시간의 **TF-IDF** 는 문서를 "어떤 단어가 얼마나 중요한가"의 표로 바꿉니다. 이 표에는 칸이 **단어 종류만큼**(수천~수만 개) 있고, 한 문장에는 그중 몇 단어만 나오니 **거의 다 0** 입니다. 이런 벡터를 **희소(sparse) 벡터**라 부릅니다.

### 비유 — 체크리스트 vs 지도 위의 점
- **희소 벡터(TF-IDF)** 는 수만 칸짜리 **단어 출석부**입니다. "총격전"에 체크가 있으면 1, 없으면 0. 두 문장이 비슷하려면 **같은 칸에 체크**가 있어야 합니다. 그래서 뜻은 같아도 **다른 단어**를 쓰면 겹치는 칸이 없어 "안 비슷하다"가 됩니다.
- **밀집(dense) 벡터(임베딩)** 는 문장을 **의미 공간의 한 점**으로 찍습니다. 칸이 768개뿐이고 **모든 칸에 실수 값**이 들어갑니다. 뜻이 비슷한 문장은 **가까운 점**이 됩니다 — 단어가 겹치지 않아도요.

### 한눈에 비교
| | 희소 벡터 (BoW·TF-IDF) | 밀집 벡터 (임베딩) |
|---|---|---|
| 길이 | 단어 종류만큼(수천~수만) | 고정(예: 768) |
| 값 | 대부분 0 | 모든 칸에 실수 |
| 비슷함의 기준 | **같은 단어**가 겹쳐야 | **뜻**이 가까우면 |
| 못 잡는 것 | 동의어·바꿔 쓴 표현 | (잘 잡음) |

> 임베딩은 대량의 문장으로 **미리 학습된 모델**이 만들어 줍니다. 우리는 그 모델을 **불러다 쓰기만** 하면 됩니다. 다음 절에서 실제로 만들어 봅니다.

아래는 우리 데이터(영화 리뷰 15건)를 실제로 두 방식으로 재어 본 결과입니다. **겹치는 낱말이 하나도 없는 두 로맨스 리뷰**를 TF-IDF 는 0.000(완전 남남)으로, 임베딩은 0.724(꽤 비슷)로 봅니다.

<img src="images/희소벡터_vs_밀집벡터.png" width="880">

> 여기서 "겹치는 낱말이 없다"는 **띄어쓰기로 자른 낱말** 기준입니다. 두 문장에는 "연인**의**"·"연인**으로**" 가 있어 사람 눈엔 '연인'이 겹쳐 보이지만, TF-IDF 는 조사가 붙은 두 낱말을 **서로 다른 단어**로 셉니다(11일차에서 본 형태소 분석이 필요한 이유가 이것입니다). 희소 벡터의 한계가 이렇게 두 겹입니다.

> 말로만 듣지 말고 **직접 재 봅시다.** 11일차에 배운 `TfidfVectorizer` 를 그대로 씁니다.

In [ ]:
# 11일차의 TF-IDF 로 재보고, 오늘 배울 임베딩과 비교한다 (리뷰 6번 <-> 8번)
from sklearn.feature_extraction.text import TfidfVectorizer

docs = df['review'].tolist()
tfidf_m = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b').fit_transform(docs)
print(f'TF-IDF 행렬 모양: {tfidf_m.shape}   (리뷰 15건 × 단어 종류만큼)')

n_cells = tfidf_m.shape[0] * tfidf_m.shape[1]   # 전체 칸 수
zero_ratio = 1 - tfidf_m.nnz / n_cells          # 0 인 칸의 비율
print(f'0 인 칸의 비율: {zero_ratio:.1%}')

print('\n[6번]', docs[6])
print('[8번]', docs[8])
print('\nTF-IDF 유사도:', round(cosine_similarity(tfidf_m[6], tfidf_m[8])[0][0], 3),
      '  <- 겹치는 낱말이 없어 0')

임베딩으로 잰 값(**0.724**)은 다음 절에서 `model.encode` 를 배운 뒤 바로 확인합니다. **같은 두 문장인데 0.000 과 0.724** — 이 차이가 오늘 배울 내용 전부의 출발점입니다.

### ✅ 바로 확인 퀴즈

**1.** "긴장감 넘치는 액션"과 "손에 땀을 쥐게 하는 전투"는 뜻이 비슷합니다. **TF-IDF**(희소 벡터)로 이 둘의 유사도를 재면 왜 낮게 나올까요?

<details><summary>정답 보기</summary>

두 문장에 **겹치는 단어가 거의 없기** 때문입니다. TF-IDF 는 같은 단어가 나와야 비슷하다고 보는데, 여기선 "긴장감·액션" vs "땀·전투"로 단어가 달라 겹치는 칸이 없습니다. 뜻은 같아도 유사도가 낮게 나옵니다.

</details>

**2.** 임베딩(밀집 벡터)이 위 문제를 해결할 수 있는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

임베딩은 문장을 **의미 공간의 점**으로 바꿉니다. 뜻이 비슷하면 단어가 달라도 **가까운 점**이 되므로, "긴장감 넘치는 액션"과 "손에 땀을 쥐게 하는 전투"가 서로 가깝게 놓입니다.

</details>

---
# 2. HuggingFace 임베딩 만들기 — 문장을 벡터로

## 왜 필요할까요?
임베딩을 직접 학습하려면 방대한 데이터와 시간이 듭니다. 다행히 **이미 학습된 모델**을 누구나 내려받아 쓸 수 있습니다. 대표 창구가 **HuggingFace** 이고, 그중 문장을 벡터로 바꾸는 라이브러리가 **`sentence-transformers`** 입니다. 우리는 한국어에 강한 **`jhgan/ko-sroberta-multitask`** 모델을 씁니다.

### 문법
- **`model = SentenceTransformer('jhgan/ko-sroberta-multitask')`** — 모델을 불러옵니다(맨 위에서 준비함).
- **`model.encode(문장리스트)`** → `(문장 수, 768)` 모양의 벡터 배열. 문장 하나당 **768개 숫자**.
- **`df['열이름'].tolist()`** — 판다스 열(시리즈)을 **파이썬 리스트**로 바꿉니다. `encode` 는 문자열이나 **문자열 리스트**만 받아서, 시리즈를 그대로 넣으면 오류가 납니다. 앞으로 `.tolist()` 가 붙어 있으면 **거의 이 이유**입니다.

> 768이라는 숫자는 이 모델이 문장의 뜻을 담기 위해 쓰는 **좌표의 개수**입니다. 모델마다 다릅니다.

## 이 모델 안에서는 무슨 일이 일어나나 — BERT 와 SBERT
모델 이름 `ko-**s**roberta` 의 **s** 는 **Sentence** 입니다. 이 계열이 어떻게 만들어졌는지 한 줄기로 짚어 두면, 앞으로 만날 임베딩 모델 이름을 읽을 수 있습니다.

- **BERT** — 문장을 **토큰(단어 조각)** 으로 쪼개고, 토큰마다 앞뒤 문맥을 반영한 벡터를 만듭니다. 그래서 BERT 의 출력은 **문장 하나당 벡터 하나가 아니라 토큰 개수만큼의 벡터**입니다. 문장끼리 비교하려면 이걸 **하나로 합쳐야** 합니다.
- **풀링(pooling)** — 토큰 벡터들을 **평균 내어** 문장 벡터 하나로 만드는 단계입니다. 토큰이 몇 개든 결과는 항상 **768개 숫자 하나**로 고정됩니다. 우리가 본 `(문장 수, 768)` 이 여기서 나옵니다.
- **SBERT (Sentence-BERT)** — BERT + 풀링을 붙인 뒤, **뜻이 비슷한 문장쌍은 가깝게 / 다른 쌍은 멀게** 되도록 **추가로 학습**시킨 모델입니다. 그래서 코사인 유사도를 그대로 믿고 쓸 수 있습니다. `sentence-transformers` 라이브러리가 바로 이 계열을 다룹니다.

> **차원(768)이 왜 그 값인가** — 풀링 전 BERT 토큰 벡터의 크기(`hidden_size`)가 그대로 문장 벡터의 크기가 됩니다. 작은 모델은 384, 큰 모델은 1024·1536 을 쓰기도 합니다. **차원이 크면 표현력이 늘지만 저장 공간과 검색 비용도 함께 늘어난다** — 6절에서 모델을 고를 때 다시 만날 기준입니다.

In [ ]:
# 리뷰 15건을 한꺼번에 임베딩 — encode 는 (문장 수, 768) 배열을 준다
reviews = df['review'].tolist()
embeddings = model.encode(reviews)

print('임베딩 배열 모양:', embeddings.shape, '  (리뷰 15건 × 768차원)')
print('\n첫 리뷰:', reviews[0])
print('그 임베딩의 앞 8개 값:', np.round(embeddings[0][:8], 3))
print('→ 문장 하나가 768개의 실수로 바뀌었다(밀집 벡터)')

### 🖐️ 함께 따라하기 — 내 문장을 벡터로 바꾸기

짧은 문장 두 개를 직접 임베딩해 벡터 모양을 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) my_sentences = ['정말 무서운 공포 영화였어요', '가슴 뭉클한 감동 드라마였어요'] 를 만든다
# 2) model.encode(my_sentences) 로 벡터를 만들어 my_vecs 에 담는다
# 3) my_vecs.shape 를 출력해 (2, 768) 인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** `model.encode(['문장 A', '문장 B', '문장 C'])` 의 결과 모양(shape)은 무엇인가요?

<details><summary>정답 보기</summary>

`(3, 768)` 입니다. 문장 3개가 각각 768차원 벡터로 바뀌므로 **행 3개 · 열 768개** 배열이 됩니다.

</details>

**2.** 임베딩 벡터의 각 칸에는 어떤 값이 들어 있나요? (희소 벡터와 비교해 보세요)

<details><summary>정답 보기</summary>

**모든 칸에 실수 값**이 들어 있습니다(밀집 벡터). 대부분이 0인 희소 벡터(TF-IDF)와 달리, 임베딩은 768칸 전부에 문장의 뜻을 나눠 담습니다.

</details>

---
# 3. 코사인 유사도 — 두 문장이 얼마나 비슷한가

## 왜 필요할까요?
문장을 벡터(점)로 바꿨으니, 이제 **두 점이 얼마나 비슷한지** 숫자로 재야 합니다. 임베딩에서는 두 벡터가 **같은 방향**을 가리키는지를 보는 **코사인 유사도**를 씁니다.

### 비유 — 방향으로 뜻을 비교
두 벡터를 원점에서 뻗은 **화살표**라고 봅니다. 화살표가 **같은 방향**이면 뜻이 비슷하고, **직각**이면 무관합니다. 코사인 유사도는 두 화살표 **사이 각도**를 −1~1 로 바꾼 값입니다 (1=같은 방향, 0=무관). 임베딩끼리 비교할 땐 보통 0~1 범위로 나옵니다.

<img src="images/코사인유사도_각도.png" width="860">

> 그런데 "각도를 잰다"는 말은 아직 비유입니다. **실제로 어떤 계산이 일어나는지**를 3-1 절에서 수식으로 먼저 확인하고, 그 다음에 라이브러리 함수를 씁니다.

---
## 3-1. 벡터와 수식 — 코사인 유사도는 무엇을 계산하나

### 벡터란 무엇인가
**벡터는 숫자를 정해진 순서로 늘어놓은 것**입니다. 그게 전부입니다.

$$ \vec{a} = (a_1,\; a_2,\; \ldots,\; a_n) $$

칸의 개수 $n$ 을 **차원(dimension)** 이라 부릅니다. 우리가 만든 임베딩은 $n = 768$ 이니 **768개의 실수를 순서대로 늘어놓은 것**이고, 그것을 **768차원 공간에 찍은 점 하나**(또는 원점에서 그 점까지 뻗은 **화살표**)로 봅니다.

<img src="images/의미공간_비유.png" width="820">

> 문장 하나하나가 **의미 공간이라는 지도 위의 점**이 되고, 뜻이 비슷한 문장들은 저절로 한 구역에 모입니다. 각 점을 **원점에서 뻗은 화살표**로 보면, 두 문장의 유사도는 **두 화살표 사이의 각**이 됩니다 — 우리가 계산할 것이 바로 이 각입니다.

768차원은 그릴 수 없으니 **2차원으로 줄여서** 계산법을 익힙니다. 칸이 2개든 768개든 **계산 방법은 글자 그대로 똑같습니다.**

<img src="images/벡터연산_수식.png" width="900">

### 필요한 연산은 딱 두 개 — 내적과 길이

**① 내적(dot product)** — 두 벡터의 **같은 칸끼리 곱해서 전부 더한 값**입니다. 곱셈·덧셈만 있고, 결과는 **숫자 하나**입니다.

$$ \vec{a} \cdot \vec{b} \;=\; \sum_{i=1}^{n} a_i b_i \;=\; a_1b_1 + a_2b_2 + \cdots + a_nb_n $$

$\vec{a}=(3,4)$, $\vec{b}=(4,3)$ 이면 $\vec{a}\cdot\vec{b} = 3{\times}4 + 4{\times}3 = 24$ 입니다.

**② 길이(노름, norm)** — 화살표가 얼마나 긴가. 각 칸을 제곱해 더하고 제곱근을 씌웁니다(피타고라스 정리를 $n$ 개 칸으로 늘린 것입니다).

$$ \lVert \vec{a} \rVert \;=\; \sqrt{\sum_{i=1}^{n} a_i^2} \;=\; \sqrt{a_1^2 + a_2^2 + \cdots + a_n^2} $$

$\vec{a}=(3,4)$ 의 길이는 $\sqrt{3^2+4^2} = \sqrt{25} = 5$ 입니다.

### 코사인 유사도 = 내적 ÷ (길이 × 길이)

$$ \cos\theta \;=\; \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \; \lVert \vec{b} \rVert } \;=\; \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum a_i^2} \; \sqrt{\sum b_i^2}} $$

분자는 내적, 분모는 두 길이의 곱입니다. **길이로 나누는 순간 길이의 영향이 사라지고 방향만 남습니다** — 그래서 "각도를 잰다"고 말하는 것입니다. 위 예시로는

$$ \cos\theta = \frac{24}{5 \times 5} = \frac{24}{25} = 0.96 \quad (\theta \approx 16.26^\circ) $$

| $\cos\theta$ | 두 화살표의 각 | 뜻 |
|---|---|---|
| **1** | 0도 | 완전히 같은 방향 |
| **0.96** | 약 16도 | 거의 같은 방향 |
| **0** | 90도(직각) | 서로 무관 |
| **−1** | 180도 | 정반대 방향 |

> 값이 $-1 \le \cos\theta \le 1$ 범위를 벗어날 수 없는 이유도 이 식에 있습니다. 내적은 아무리 커도 두 길이의 곱을 넘지 못하기 때문입니다(코시-슈바르츠 부등식).

> 아래 셀에서 **라이브러리 없이 파이썬 기본 문법만으로** 이 식을 그대로 옮겨 계산하고, `cosine_similarity` 가 내놓는 값과 같은지 확인합니다.

In [ ]:
# 수식을 그대로 파이썬으로 옮겨 본다 — 2차원 벡터 a, b
a = [3, 4]
b = [4, 3]

# ① 내적: 같은 칸끼리 곱해서 전부 더한다
dot_ab = sum(a[i] * b[i] for i in range(len(a)))

# ② 길이(노름): 각 칸을 제곱해 더하고 제곱근
norm_a = sum(v ** 2 for v in a) ** 0.5
norm_b = sum(v ** 2 for v in b) ** 0.5

# ③ 코사인 유사도 = 내적 / (길이 x 길이)
cos_ab = dot_ab / (norm_a * norm_b)

print('내적 a·b      :', dot_ab)
print('길이 |a|, |b| :', norm_a, norm_b)
print('코사인 유사도  :', cos_ab)

# 라이브러리가 주는 값과 같은지 확인 — 같은 식을 계산하니 같아야 한다
print('\ncosine_similarity :', float(cosine_similarity([a], [b])[0][0]))

### 왜 굳이 길이로 나누나 — 나누지 않으면 어떻게 되나

$\vec{d} = (6,8)$ 을 봅시다. 이것은 $\vec{a}=(3,4)$ 와 **방향이 완전히 같고 길이만 2배**인 벡터입니다($\vec{d} = 2\vec{a}$). 문장으로 치면 **같은 말을 두 배 길게 쓴 문장**에 해당합니다.

<img src="images/코사인_길이무시.png" width="900">

- **내적**만 보면 $\vec{a}\cdot\vec{d} = 50$ 으로 $\vec{a}\cdot\vec{b}=24$ 보다 훨씬 큽니다.
- **유클리드 거리**로 보면 $\vec{d}$ 는 5.00 만큼 떨어져 있어 $\vec{b}$(1.41)보다 **더 멀어** 보입니다.
- **코사인**만이 $\cos\theta = 1.00$ — "방향이 똑같다 = 같은 뜻" 이라고 제대로 답합니다.

**길이는 대개 '문장이 길다'를 뜻하지 '뜻이 다르다'를 뜻하지 않습니다.** 그래서 문장 임베딩 비교의 기본값이 코사인입니다.

> $\vec{c} = (-4,3)$ 은 $\vec{a}$ 와 **직각**이라 내적이 정확히 0 이고 코사인도 0 입니다 ($3{\times}(-4) + 4{\times}3 = 0$). "완전히 무관"의 수학적 모습이 바로 이것입니다.

In [ ]:
# a 를 기준으로 b · c · d 를 세 가지 척도로 재어 표로 본다
import math

def vec_dot(u, v):
    return sum(u[i] * v[i] for i in range(len(u)))

def vec_norm(u):
    return math.sqrt(vec_dot(u, u))

def vec_cos(u, v):
    return vec_dot(u, v) / (vec_norm(u) * vec_norm(v))

def vec_euclid(u, v):
    return math.sqrt(sum((u[i] - v[i]) ** 2 for i in range(len(u))))

a = [3, 4]
rows = []
for name, v in [('b = (4, 3)', [4, 3]), ('d = (6, 8) = 2a', [6, 8]), ('c = (-4, 3)', [-4, 3])]:
    rows.append({'상대 벡터': name, '내적': vec_dot(a, v), '길이': round(vec_norm(v), 2),
                 '유클리드 거리': round(vec_euclid(a, v), 2), '코사인': round(vec_cos(a, v), 2)})

display(pd.DataFrame(rows))
print('d 는 a 와 방향이 같아 코사인 1.00 — 내적·거리는 길이에 흔들린다')

### 768차원이어도 똑같습니다

지금까지 칸이 2개였을 뿐, 식에는 **칸의 개수 $n$ 이 어디에도 제한으로 들어 있지 않습니다.** $\sum_{i=1}^{n}$ 의 $n$ 을 768 로 바꾸면 그대로 문장 임베딩의 코사인 유사도가 됩니다.

실제 리뷰 임베딩으로 **직접 계산한 값**과 `cosine_similarity` 의 값이 같은지 확인해 봅시다. 이 둘이 같다는 것을 한 번 보고 나면, 앞으로 `cosine_similarity` 를 **믿고 쓸 수 있습니다.**

> **numpy 의 두 도구를 여기서 처음 씁니다.**
> - **`u @ v`** — 두 벡터의 **내적**. 위에서 만든 `vec_dot()` 과 하는 일이 같습니다. `@` 는 파이썬의 **행렬 곱 연산자**로, 벡터 두 개에 쓰면 내적 하나(숫자)가 나옵니다.
> - **`np.linalg.norm(u)`** — 그 벡터의 **길이**. `vec_norm()` 과 같습니다.
> 
> 둘 다 768칸을 훨씬 빠르게 처리합니다. (`@` 는 **numpy 배열**에만 쓸 수 있습니다 — 파이썬 리스트에는 안 되니, 리스트로 직접 계산할 땐 위의 `vec_dot()` 처럼 `sum` 을 쓰세요.)

In [ ]:
# 768차원 임베딩 두 개를 꺼내 수식대로 직접 계산한다
u, v = embeddings[0], embeddings[2]      # 리뷰 0번(액션) 과 2번(액션)
print('벡터 칸 수(차원):', len(u))

dot_uv = float(u @ v)                            # 칸마다 곱해서 전부 더하기(내적)
norm_u = float(np.linalg.norm(u))                # 제곱합의 제곱근
norm_v = float(np.linalg.norm(v))
cos_manual = dot_uv / (norm_u * norm_v)          # 내적 / (길이 x 길이)

print(f'내적          : {dot_uv:.4f}')
print(f'길이 |u|, |v| : {norm_u:.4f}, {norm_v:.4f}')
print(f'직접 계산한 코사인      : {cos_manual:.6f}')

cos_sklearn = float(cosine_similarity([u], [v])[0][0])
print(f'cosine_similarity 값   : {cos_sklearn:.6f}')
print('두 값이 같은가?', np.isclose(cos_manual, cos_sklearn))

### 🖐️ 함께 따라하기 — 코사인 유사도 함수를 직접 만들어 보기

위 수식을 함수 하나로 만들어, **내가 고른 문장 두 개**에 적용해 봅니다. 라이브러리가 뒤에서 무슨 일을 하는지 손으로 한 번 재현해 보는 것이 목적입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) my_cosine(u, v) 함수를 만든다 — @ 와 np.linalg.norm 으로 '내적 / (길이 x 길이)' 를 계산해 반환
# 2) sents = ['노트북이 너무 느려서 답답해요', '컴퓨터 속도가 느려 불편합니다'] 를 만든다
# 3) model.encode(sents) 로 벡터를 만들어 my_vecs 에 담는다
# 4) my_cosine(my_vecs[0], my_vecs[1]) 값과 cosine_similarity 값을 함께 출력해 같은지 확인한다

### ✅ 바로 확인 퀴즈

**1.** $\vec{u} = (1, 2, 2)$ 의 길이 $\lVert \vec{u} \rVert$ 는 얼마인가요?

<details><summary>정답 보기</summary>

**3** 입니다. $\sqrt{1^2 + 2^2 + 2^2} = \sqrt{1+4+4} = \sqrt{9} = 3$.

</details>

**2.** 두 벡터의 **내적이 0** 으로 나왔습니다. 코사인 유사도는 얼마이고, 두 문장은 어떤 관계인가요?

<details><summary>정답 보기</summary>

코사인 유사도도 **0** 입니다 — 분자가 0 이면 무엇으로 나눠도 0 이기 때문입니다. 두 벡터가 **직각(90도)** 이라는 뜻이고, 의미상으로는 **서로 무관한 문장**으로 봅니다.

</details>

**3.** 어떤 문장을 그대로 **두 번 이어 붙여** 더 긴 문장을 만들었더니 벡터의 길이가 커졌습니다. 원래 문장과의 **코사인 유사도**는 왜 크게 변하지 않을까요?

<details><summary>정답 보기</summary>

코사인은 분모에서 **두 길이를 나눠 없애기** 때문입니다. 방향(뜻)이 그대로면 길이가 몇 배가 되어도 값이 거의 그대로입니다. 위 $\vec{d} = 2\vec{a}$ 예시처럼 **길이가 2배가 되면 내적도 2배**가 되지만, 코사인은 1.00 그대로였던 것과 같은 이치입니다.

</details>

---
### 이제 라이브러리로 — 리뷰 15건을 한 번에

### 문법
- **`cosine_similarity(A, B)`** → A 의 각 행과 B 의 각 행 사이 유사도 **행렬**을 돌려줍니다.
- 한 쌍만 재려면 `cosine_similarity([벡터1], [벡터2])[0][0]` 로 값 하나를 꺼냅니다.

> 참고: 코사인은 **길이를 무시하고 방향만** 봅니다. 그래서 벡터를 **길이 1로 맞추면(L2 정규화)** 길이 차이가 사라져 방향 비교가 더 안정적입니다. 정규화해서 내놓는 모델도 있고 아닌 모델도 있는데, **우리가 쓰는 모델은 정규화하지 않은 벡터**를 줍니다 — 3-2 절에서 직접 확인합니다.

> 아래에서 **같은 장르** 리뷰끼리(액션↔액션)와 **다른 장르** 리뷰끼리(액션↔로맨스)의 유사도를 실제로 재서, 임베딩이 뜻의 차이를 담는지 확인합니다.

In [ ]:
# 리뷰 사이 코사인 유사도 행렬 — (15, 15). 값이 클수록 뜻이 비슷하다
sim_matrix = cosine_similarity(embeddings, embeddings)
print('유사도 행렬 모양:', sim_matrix.shape)

# 몇 쌍을 직접 비교 — 같은 장르는 높고, 다른 장르는 낮아야 한다
print('\n[같은 장르 vs 다른 장르]')
print('리뷰0(액션) ↔ 리뷰2(액션) :', round(sim_matrix[0, 2], 3))
print('리뷰0(액션) ↔ 리뷰5(로맨스):', round(sim_matrix[0, 5], 3))
print('리뷰0(액션) ↔ 리뷰11(코미디):', round(sim_matrix[0, 11], 3))
print('\n같은 장르(액션↔액션)가 다른 장르보다 뚜렷하게 높다')

In [ ]:
# 장르 안(within) 평균 vs 장르 밖(between) 평균 유사도를 비교
genres = df['genre'].values
within, between = [], []
for i in range(len(reviews)):
    for j in range(i + 1, len(reviews)):
        if genres[i] == genres[j]:
            within.append(sim_matrix[i, j])
        else:
            between.append(sim_matrix[i, j])
within_mean = np.mean(within)
between_mean = np.mean(between)
print(f'같은 장르 리뷰끼리 평균 유사도: {within_mean:.3f}')
print(f'다른 장르 리뷰끼리 평균 유사도: {between_mean:.3f}')
print('→ 같은 장르가 약 2배 더 비슷 — 임베딩이 장르(뜻)를 잘 담았다')

### 🖐️ 함께 따라하기 — 두 문장의 유사도 직접 재보기

뜻이 비슷한 문장쌍과 동떨어진 문장쌍의 유사도를 직접 재서 비교해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pair = ['통쾌한 액션이 넘치는 영화', '박진감 있는 전투 장면', '따뜻한 사랑 이야기'] 를 만든다
# 2) model.encode(pair) 로 벡터 3개를 만들어 pair_vecs 에 담는다
# 3) cosine_similarity(pair_vecs[0:1], pair_vecs[1:2])[0][0] (비슷한 쌍)을 소수 셋째 자리로 출력한다
# 4) cosine_similarity(pair_vecs[0:1], pair_vecs[2:3])[0][0] (동떨어진 쌍)도 출력해 비교한다

### ✅ 바로 확인 퀴즈

**1.** 코사인 유사도가 **1** 에 가까울수록 두 문장은 어떤 관계인가요?

<details><summary>정답 보기</summary>

두 벡터가 **같은 방향**을 가리켜 **뜻이 매우 비슷하다**는 뜻입니다. 반대로 0에 가까우면 서로 무관합니다.

</details>

**2.** 같은 장르 리뷰끼리의 평균 유사도가 다른 장르끼리보다 **높게** 나왔습니다. 이것이 말해 주는 것은?

<details><summary>정답 보기</summary>

임베딩이 **장르(뜻)의 차이를 제대로 담았다**는 신호입니다. 같은 장르 문장은 뜻이 가까워 벡터도 가깝고, 다른 장르 문장은 뜻이 멀어 벡터도 멀기 때문에 유사도 차이가 납니다.

</details>

---
## 3-2. 다른 재는 법 — 유클리드 거리와 내적

### 왜 알아둘까요?
두 벡터의 가까움을 재는 방법은 코사인 하나가 아닙니다. 자주 쓰이는 것이 셋입니다. 벡터 DB 를 만들 때도 "거리 척도(metric)를 무엇으로 할까"를 반드시 고르게 되므로, **셋이 어떻게 다르고 왜 임베딩에서는 코사인이 기본인지**를 지금 확실히 해 둡니다.

| 재는 법 | 무엇을 보나 | 값의 방향 | 함수 |
|---|---|---|---|
| **코사인 유사도** | 두 화살표의 **각도**(길이 무시) | **클수록** 비슷 (−1~1) | `cosine_similarity` |
| **유클리드 거리** | 두 점 사이의 **직선 거리** | **작을수록** 비슷 (0~∞) | `euclidean_distances` |
| **내적(dot product)** | 각도 **와 길이**를 함께 | **클수록** 비슷 (제한 없음) | `A @ B` |

> **부호가 반대인 것에 주의하세요.** 코사인·내적은 **큰 값**이 비슷한 것이고, 유클리드는 **거리**라서 **작은 값**이 비슷한 것입니다. 정렬 방향을 반대로 쓰면 가장 안 비슷한 문서를 top1 으로 뽑게 됩니다.

### 수식으로 나란히 놓고 보기
3-1 절에서 만든 내적·노름만 있으면 셋을 모두 쓸 수 있습니다.

$$ \text{내적} \;=\; \vec{a} \cdot \vec{b} \;=\; \sum_{i=1}^{n} a_i b_i $$

$$ \text{유클리드 거리} \;=\; \lVert \vec{a} - \vec{b} \rVert \;=\; \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2} $$

$$ \text{코사인 유사도} \;=\; \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \; \lVert \vec{b} \rVert} $$

차이는 **길이를 어떻게 다루는가** 하나뿐입니다. 내적은 길이를 그대로 곱해 넣고, 유클리드는 길이 차이를 거리로 떠안으며, 코사인만 길이로 나눠 없앱니다.

### 문법
- **`euclidean_distances(A, B)`** → A·B 행 사이의 **거리** 행렬 (맨 위 셀에서 이미 import 했습니다).
- **`벡터A @ 벡터B`** → 내적(스칼라 하나). 3-1 절에서 쓴 그 연산자입니다.
- **`np.linalg.norm(벡터)`** → 그 벡터의 **길이**.

> 아래에서 같은 세 쌍을 **세 가지 방법으로 모두** 재어, 순서가 같은지 다른지 직접 확인합니다.

In [ ]:
# 같은 세 쌍을 코사인 · 유클리드 거리 · 내적으로 모두 재어 본다
for j, name in [(2, '액션'), (5, '로맨스'), (11, '코미디')]:
    cos_v = float(cosine_similarity(embeddings[0:1], embeddings[j:j+1])[0][0])
    euc_v = float(euclidean_distances(embeddings[0:1], embeddings[j:j+1])[0][0])
    dot_v = float(embeddings[0] @ embeddings[j])
    print(f'리뷰0(액션) ↔ 리뷰{j}({name}) : '
          f'코사인 {cos_v:.3f} | 유클리드 거리 {euc_v:.2f} | 내적 {dot_v:.1f}')

print('\n→ 액션↔액션은 코사인·내적이 가장 크고, 유클리드 거리는 가장 작다(부호가 반대)')

### 그럼 셋 중 무엇을 써도 같은 결과일까 — 벡터의 **길이**가 갈림길입니다

코사인은 길이를 나눠 없애지만, **유클리드와 내적은 길이에 영향을 받습니다.** 그래서 우리 모델처럼 벡터 길이가 제각각이면 **순위가 어긋날 수 있습니다.** 아래에서 길이를 먼저 확인하고, 검색 순위를 세 방법으로 비교해 봅니다.

In [ ]:
# 1) 이 모델의 벡터 길이는 1 로 맞춰져 있을까?
lengths = np.linalg.norm(embeddings, axis=1)
print('벡터 길이(앞 5개):', np.round(lengths[:5], 2))
print('→ 1 이 아니다. 이 모델은 정규화된 벡터를 주지 않는다(모델마다 다르다)')

# 2) 같은 질문을 세 방법으로 검색해 순위를 비교
q_vec = model.encode(['박진감 넘치는 싸움 장면'])
cos_s = cosine_similarity(q_vec, embeddings)[0]
euc_d = euclidean_distances(q_vec, embeddings)[0]

rank_cos = np.argsort(-cos_s).tolist()   # 유사도는 큰 순
rank_euc = np.argsort(euc_d).tolist()    # 거리는 작은 순
print('\n코사인 순위 :', rank_cos)
print('유클리드 순위:', rank_euc)
print('두 순위가 같은가?', rank_cos == rank_euc)

### 길이를 1 로 맞추면(L2 정규화) 셋이 한 줄로 정리됩니다

**L2 정규화**란 벡터를 **자기 길이로 나누는 것**입니다. 방향은 그대로 두고 길이만 1 로 만듭니다.

$$ \hat{a} \;=\; \frac{\vec{a}}{\lVert \vec{a} \rVert} \qquad\Longrightarrow\qquad \lVert \hat{a} \rVert = 1 $$

길이가 1 이면 코사인 식의 **분모가 1×1 = 1** 이 되어 사라집니다. 그러면

- **내적 = 코사인 유사도** — $\hat{a} \cdot \hat{b} = \dfrac{\vec{a}\cdot\vec{b}}{\lVert \vec{a}\rVert \lVert \vec{b}\rVert} = \cos\theta$ (완전히 같은 값이 됩니다)
- **유클리드 거리 $= \sqrt{2 - 2\cos\theta}$** — 제곱을 풀어 보면 바로 나옵니다.

$$ \lVert \hat{a} - \hat{b} \rVert^2 \;=\; \lVert \hat{a} \rVert^2 - 2\,\hat{a}\cdot\hat{b} + \lVert \hat{b} \rVert^2 \;=\; 1 - 2\cos\theta + 1 \;=\; 2 - 2\cos\theta $$

  코사인이 클수록 거리가 작아지는 **단조 관계**이므로 **순위가 항상 일치**합니다.

즉 **정규화만 해 두면 셋 중 무엇을 써도 검색 결과가 같아집니다.** 그래서 실무에서는 임베딩을 저장할 때 미리 정규화해 두고, 계산이 가장 싼 **내적**을 쓰는 경우가 많습니다. 정규화를 하지 않았다면 **길이에 흔들리지 않는 코사인**이 안전한 기본값입니다.

In [ ]:
# 모든 벡터를 길이 1 로 맞춘다(L2 정규화)
emb_unit = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
q_unit = q_vec / np.linalg.norm(q_vec)
print('정규화 후 길이(앞 5개):', np.round(np.linalg.norm(emb_unit, axis=1)[:5], 2))

dot_unit = (q_unit @ emb_unit.T)[0]                       # 정규화 후 내적
euc_unit = euclidean_distances(q_unit, emb_unit)[0]        # 정규화 후 거리

print('\n정규화 후 내적 == 코사인 인가?', np.allclose(dot_unit, cos_s, atol=1e-5))
print('정규화 후 유클리드 순위 == 코사인 순위 인가?',
      np.argsort(euc_unit).tolist() == np.argsort(-cos_s).tolist())

### 🖐️ 함께 따라하기 — 내 문장쌍을 세 가지로 재보기

위 시연은 **리뷰 데이터**로 셋을 비교했습니다. 이번엔 **내가 쓴 문장 두 개**로 직접 재어, 세 값이 어떻게 나오는지 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) trio = ['배송이 하루 만에 왔어요', '주문한 다음 날 바로 받았어요', '색상이 사진과 달라요'] 를 만든다
# 2) model.encode(trio) 로 벡터를 만들어 trio_vecs 에 담는다
# 3) 0번<->1번(뜻이 비슷) 과 0번<->2번(동떨어짐) 을
#    cosine_similarity / euclidean_distances / @ 세 가지로 각각 재어 출력한다
# 4) 비슷한 쌍이 '코사인·내적은 크고 유클리드 거리는 작은지' 확인한다

### ✅ 바로 확인 퀴즈

**1.** 유클리드 거리로 가장 비슷한 문서를 찾으려면 값이 **큰** 순으로 정렬해야 할까요, **작은** 순으로 정렬해야 할까요?

<details><summary>정답 보기</summary>

**작은 순**입니다. 유클리드는 **거리**라서 0 에 가까울수록 두 점이 붙어 있다는 뜻입니다. 코사인·내적(클수록 비슷)과 정렬 방향이 반대라는 점을 꼭 기억하세요.

</details>

**2.** 벡터를 모두 **길이 1** 로 맞추면 내적은 무엇과 같아지나요?

<details><summary>정답 보기</summary>

**코사인 유사도**와 같아집니다. 코사인은 내적을 두 벡터의 길이로 나눈 값인데, 길이가 둘 다 1 이면 나눌 것이 없어져 내적이 곧 코사인이 됩니다. 이때 유클리드 거리도 코사인과 **순위가 일치**합니다.

</details>

**3.** 정규화를 하지 않은 임베딩에서 **코사인**을 기본으로 권하는 이유는?

<details><summary>정답 보기</summary>

코사인만 **길이의 영향을 받지 않기** 때문입니다. 문장이 길거나 표현이 많으면 벡터가 길어지는데, 유클리드·내적은 그 길이를 함께 반영해 "문장이 길다"를 "뜻이 다르다/같다"로 잘못 읽을 수 있습니다.

</details>

---
# 4. 의미 검색 — 질문에 가장 가까운 문서 찾기

## 왜 필요할까요?
검색창에 "박진감 넘치는 싸움 장면"이라고 쳤을 때, **똑같은 단어가 없어도** 뜻이 맞는 리뷰를 찾아 주면 좋겠죠. 이것이 **의미 검색(semantic search)** 입니다. 키워드가 겹치는지가 아니라 **뜻이 가까운지**로 찾습니다.

### 원리 (세 단계)
1. 찾고 싶은 문서들(**corpus**)을 미리 임베딩해 둔다. (우리는 리뷰 15건을 이미 임베딩했습니다.)
2. **질문(query)** 도 같은 모델로 임베딩한다.
3. 질문 벡터와 모든 문서 벡터의 **코사인 유사도**를 재서, 값이 큰 순으로 **top-k** 를 고른다.

### 문법
- **`np.argsort(-유사도배열)`** — 유사도를 **내림차순**으로 정렬한 **인덱스**를 준다(앞에서 k개가 top-k).

> 아래 쿼리 "박진감 넘치는 싸움 장면"에는 리뷰에 그대로 나온 단어가 거의 없습니다. 그래도 임베딩은 **뜻이 맞는 액션 리뷰**를 찾아냅니다.

In [ ]:
# 질문(쿼리)을 임베딩해 리뷰 15건 중 뜻이 가장 가까운 top3 를 찾는다
query = '박진감 넘치는 싸움 장면'
query_vec = model.encode([query])

# 쿼리 vs 모든 리뷰의 코사인 유사도 (길이 15 배열)
scores = cosine_similarity(query_vec, embeddings)[0]
top3 = np.argsort(-scores)[:3]        # 내림차순 정렬 후 앞 3개

print('질문:', query)
print('\n[가장 가까운 리뷰 top3]')
for rank, i in enumerate(top3, start=1):
    print(f"  {rank}위  유사도 {scores[i]:.3f}  [{df.iloc[i]['genre']}] {reviews[i]}")
print('\n→ 쿼리 단어가 리뷰에 그대로 없어도, 뜻이 맞는 액션 리뷰가 뽑힌다')

### 🖐️ 함께 따라하기 — 다른 질문으로 검색해 보기

이번엔 **로맨스**를 겨냥한 질문으로 top3 를 찾아봅니다. 위 시연의 흐름을 그대로 따라 하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) my_query = '설레는 첫사랑 이야기' 로 질문을 정한다
# 2) model.encode([my_query]) 로 질문 벡터 my_query_vec 을 만든다
# 3) cosine_similarity(my_query_vec, embeddings)[0] 로 유사도 배열 my_scores 를 만든다
# 4) np.argsort(-my_scores)[:3] 으로 top3 인덱스를 구해, 장르와 리뷰를 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** 의미 검색에서 **질문(query)** 을 임베딩할 때, 문서를 임베딩한 모델과 같은 모델을 써야 할까요?

<details><summary>정답 보기</summary>

네, **같은 모델**을 써야 합니다. 질문과 문서가 **같은 의미 공간**에 놓여야 코사인 유사도로 거리를 비교할 수 있습니다. 다른 모델로 만든 벡터끼리는 좌표계가 달라 비교가 무의미합니다.

</details>

**2.** `np.argsort(-scores)` 처럼 앞에 **마이너스**를 붙이는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

`np.argsort` 는 기본이 **오름차순**(작은 값부터)입니다. 부호를 뒤집으면 **큰 값이 앞으로** 와서, 유사도가 **높은 순(내림차순)** 인덱스를 얻습니다. 그래서 앞에서 k개가 top-k 가 됩니다.

</details>

---
# 5. OpenAI Embeddings API — 상용 API로도 같은 일을

## 왜 알아둘까요?
지금까지는 **HuggingFace 모델을 내 컴퓨터에서** 돌려 임베딩을 만들었습니다(로컬·무료). 같은 일을 **상용 API**로도 할 수 있습니다. 대표적으로 **OpenAI Embeddings API** 는 문장을 보내면 벡터를 돌려줍니다. 결과는 똑같이 **문장→벡터**이고, 그 뒤 코사인 유사도·의미 검색은 **완전히 동일**합니다.

### 문법
- **`OpenAI()`** — 클라이언트를 만듭니다. 인증은 환경변수 `OPENAI_API_KEY`(보통 `.env` 파일)로 합니다.
- **`client.embeddings.create(model=..., input=[문장, ...])`** — 문장 목록을 보내면 문장마다 임베딩이 담긴 응답이 옵니다.
- **`response.data[i].embedding`** — i 번째 문장의 임베딩(`text-embedding-3-small` 은 **1536개** 실수 리스트).

> **키가 없어도 괜찮습니다.** 아래 셀은 키가 있으면 실제 API 를 호출하고, 없으면 호출만 건너뜁니다 — 노트북은 끝까지 돌아갑니다. API 호출에는 **요금이 붙으므로** 키를 넣을 때만 실제로 나갑니다.

### HuggingFace(로컬) vs 상용 API
| | HuggingFace `sentence-transformers` | OpenAI Embeddings API |
|---|---|---|
| 실행 위치 | **내 컴퓨터**(로컬) | OpenAI **서버**(네트워크 호출) |
| 비용 | **무료**(모델만 내려받음) | **요금**(문장 길이만큼 과금) |
| 인터넷 | 최초 다운로드 후 불필요 | 매 호출마다 필요 |
| 차원 | 모델마다(우리 모델 768) | 모델마다(3-small 1536) |
| 결과 | **문장 → 벡터** | **문장 → 벡터** (동일) |

> 어느 쪽을 쓰든 **결과는 똑같이 벡터**이고, 이후의 코사인 유사도·의미 검색·시각화는 그대로입니다. 무료로 시작하려면 HuggingFace, 최신·대규모 서비스엔 상용 API 를 고려합니다.

In [ ]:
# [제공 코드] OpenAI 키가 있는지 확인합니다(.env 의 OPENAI_API_KEY).
# 키가 없어도 이 단원은 끝까지 돌아갑니다 - 다음 셀이 호출만 건너뜁니다.
import os
from dotenv import load_dotenv

load_dotenv('.env')        # 노트북과 같은 폴더
load_dotenv('../.env')     # 한 단계 위 폴더에 있으면 그것도 읽는다
HAS_OPENAI_KEY = bool(os.getenv('OPENAI_API_KEY'))
print('OpenAI API 키:', '있음 - 아래 셀이 실제 API 를 호출합니다'
      if HAS_OPENAI_KEY else '없음 - 아래 셀은 호출을 건너뜁니다')

In [ ]:
# 같은 질문·같은 리뷰를 OpenAI Embeddings API 로도 검색해 본다
# (HuggingFace 로 한 4절과 코드 흐름이 똑같다 - 벡터를 어디서 받아오느냐만 다르다)
if HAS_OPENAI_KEY:
    from openai import OpenAI

    client = OpenAI()
    response = client.embeddings.create(
        model='text-embedding-3-small',
        input=[query] + reviews,          # 질문 1개 + 리뷰 15개를 한 번에
    )
    openai_vectors = np.array([item.embedding for item in response.data])
    print('OpenAI 임베딩 shape:', openai_vectors.shape, '(768 이 아니라 1536차원)')

    openai_scores = cosine_similarity(openai_vectors[:1], openai_vectors[1:])[0]
    for rank in np.argsort(-openai_scores)[:3]:
        print(f'  {openai_scores[rank]:.3f}  [{df["genre"][rank]}]  {reviews[rank]}')
else:
    print('키가 없어 호출을 건너뜁니다. 위 코드가 실제로 하는 일:')
    print('  1) 질문 1개 + 리뷰 15개를 보내면 문장마다 1536개 실수가 돌아온다')
    print('  2) 그 뒤 코사인 유사도 -> top3 는 4절에서 한 것과 완전히 동일하다')
    print('  3) 차원 수(768 vs 1536)만 다를 뿐, 다루는 방법은 바뀌지 않는다')

### 🖐️ 함께 따라하기 — 차원이 바뀌어도 코드는 그대로
"API 든 로컬이든 결과는 벡터라 이후 처리가 같다"는 말을 **직접 확인**해 봅니다. OpenAI 가 주는 것과 **같은 모양**(1536차원)의 배열을 만들어, 4절의 검색 코드를 **한 글자도 바꾸지 않고** 그대로 돌려 보세요. **키가 없어도 되는 따라하기**입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) rng = np.random.default_rng(0) 으로 난수 생성기를 만든다
#    (OpenAI 응답 대신 '1536차원 벡터가 온 셈'만 흉내낸다 - 실제 뜻은 담기지 않는다)
# 2) fake_vectors = rng.normal(size=(16, 1536)) 로 질문 1개 + 리뷰 15개 분량을 만든다
# 3) fake_vectors.shape 를 출력해 (16, 1536) 인지 확인한다
# 4) cosine_similarity(fake_vectors[:1], fake_vectors[1:])[0] 로 유사도를 구한다
#    -> 4절과 똑같은 코드가 1536차원에서도 그대로 돈다
# 5) np.argsort(-그 유사도)[:3] 으로 top3 를 뽑아 출력한다

### ✅ 바로 확인 퀴즈

**1.** OpenAI Embeddings API 와 HuggingFace 모델의 **결과물**은 무엇이 같은가요?

<details><summary>정답 보기</summary>

둘 다 **문장을 벡터(임베딩)로** 바꿔 줍니다. 차원 수(768 vs 1536)나 실행 위치는 다르지만, "문장→벡터"라는 결과와 이후의 코사인 유사도·의미 검색 방식은 동일합니다.

</details>

**2.** "인터넷 없이 무료로" 임베딩을 만들고 싶다면 둘 중 무엇이 적합한가요?

<details><summary>정답 보기</summary>

**HuggingFace `sentence-transformers`** 입니다. 모델을 한 번 내려받으면 인터넷 없이 로컬에서 무료로 임베딩을 만들 수 있습니다. OpenAI API 는 매 호출마다 네트워크와 요금이 듭니다.

</details>

---
# 6. 임베딩 모델 고르기 — 어디서 찾고, 무엇을 보나

## 왜 필요할까요?
지금까지는 `jhgan/ko-sroberta-multitask` 를 **주는 대로** 썼습니다. 하지만 현장에서 다룰 글은 영화 리뷰가 아닐 수도, 훨씬 길 수도, 한국어가 아닐 수도 있습니다. 그때 **어떤 모델을 쓸지 스스로 고를 수 있어야** 합니다. 모델을 바꾸는 것만으로 결과가 크게 달라지거든요.

## 어디서 찾나 — HuggingFace 모델 허브
[huggingface.co/models](https://huggingface.co/models) 에서 왼쪽 필터로 좁힙니다.

- **Tasks -> Sentence Similarity** — 문장 전체를 하나의 벡터로 만드는 모델들입니다. (`Feature Extraction` 은 더 넓은 범주라 문장 임베딩용이 아닌 것도 섞입니다.)
- **Languages -> Korean** — 한국어를 학습한 모델만 남깁니다.
- **Sort -> Most downloads** — 많이 쓰이는 것이 대체로 안전한 출발점입니다.

<img src="images/hf_모델_검색.png" width="900">

> 목록 한 줄만 봐도 꽤 많은 것을 알 수 있습니다 — **모델 크기**(0.1B)·**갱신일**·**다운로드 수**·**좋아요**. 우리가 쓰는 `jhgan/ko-sroberta-multitask` 도 여기 있습니다.

## 무엇을 보나 — 체크리스트

| 볼 것 | 왜 중요한가 | 어디서 확인 |
|---|---|---|
| **언어** | 한국어를 안 배운 모델에 한국어를 넣으면 성능이 급격히 떨어집니다 | 태그의 `Korean`·`multilingual` |
| **차원** | 클수록 표현력이 늘지만 저장 공간·검색 비용도 함께 늡니다 | 카드 설명, `config.json` 의 `hidden_size` |
| **최대 입력 길이** | **넘는 부분이 소리 없이 잘립니다** (아래에서 자세히) | `sentence_bert_config.json` 의 `max_seq_length` |
| **모델 크기** | 다운로드 용량·메모리·속도를 좌우합니다 | 카드 오른쪽 `Model size` |
| **다운로드 수·갱신일** | 많이 쓰이고 최근까지 관리되는가 하는 신뢰 신호 | 목록·카드 |
| **라이선스** | 회사 서비스에 쓰려면 상업적 이용이 가능해야 합니다 | 카드 태그 (`apache-2.0` 등) |

> **라이선스 태그가 아예 없는 모델도 많습니다** — 위 스크린샷의 `jhgan/ko-sroberta-multitask` 가 그렇습니다. "찾았는데 안 보인다"면 잘못 본 게 아니라 **표기가 없는 것**이고, 그 자체가 하나의 신호입니다. 회사 서비스에 넣을 거라면 표기가 명확한 모델을 고르거나 제작자에게 확인해야 합니다.
| **용도** | 문장 유사도용인지, 검색(질문↔문서)용인지 | 카드 본문 설명 |

<img src="images/hf_모델_카드.png" width="900">

> 모델 카드 첫 문단이 보통 가장 중요한 정보를 담습니다 — 여기서는 "maps sentences & paragraphs to a **768 dimensional** dense vector space" 라고 차원을 알려 주고, 오른쪽에 지난달 다운로드 수와 `Model size 0.1B params` 가 있습니다.

## 가장 자주 놓치는 것 — 최대 입력 길이
차원이나 다운로드 수는 눈에 잘 띄지만, **`max_seq_length` 는 카드 본문에 안 적혀 있는 경우가 많습니다.** `Files and versions` 탭의 `sentence_bert_config.json` 을 열어 보세요.

<img src="images/hf_모델_스펙.png" width="900">

**128** 입니다. 우리가 쓰는 이 모델은 **약 128토큰까지만 읽고 나머지는 버립니다.** 그런데 **에러도 경고도 나지 않습니다** — 조용히 잘린 채로 그럴듯한 벡터가 나옵니다. 긴 글(뉴스 기사·계약서·논문)을 다룰 때 결과가 이상하면 여기를 가장 먼저 의심해야 합니다.

### 잠깐 — "토큰"은 낱말이 아닙니다 (서브워드)
128**토큰**이라니, 낱말 128개까지라는 뜻일까요? 아닙니다. 모델은 문장을 **서브워드(subword)** 라는 더 잘게 쪼갠 조각으로 나눕니다. 자주 쓰는 말은 통째로 한 조각이지만, 드문 말은 여러 조각으로 갈립니다 — 예를 들어 `카체이싱이` 는 `카`·`##체`·`##이`·`##싱`·`##이` 로 **5조각**이 됩니다(`##` 은 "앞 조각에 이어 붙는다"는 표시입니다).

그래서 **낱말 수보다 토큰 수가 대체로 많습니다.** 아래에서 직접 세어 보면, 낱말 6개짜리 리뷰가 토큰으로는 17개가 됩니다 — 128토큰은 생각보다 훨씬 짧습니다.

### 문법
- **`model.max_seq_length`** → 이 모델이 읽는 최대 토큰 수.
- **`model.get_sentence_embedding_dimension()`** → 임베딩 차원.
- **`model.tokenizer.tokenize(문장)`** → 그 문장이 실제로 몇 조각(토큰)으로 쪼개지는지 확인합니다.

> 아래에서 **정말 잘리는지** 직접 확인합니다. 128토큰이 넘도록 길게 쓴 뒤 **맨 뒤에만** 정반대 표현을 붙인 두 문장의 유사도를 재 봅니다.

In [ ]:
# 이 모델의 스펙을 코드로 확인 (허브 화면과 같은 값)
print('최대 입력 길이:', model.max_seq_length, '토큰')
print('임베딩 차원   :', model.get_sentence_embedding_dimension())

In [ ]:
# 낱말 수와 토큰 수는 다르다 — 직접 세어 본다
sample = df.loc[0, 'review']
pieces = model.tokenizer.tokenize(sample)
print('문장    :', sample)
print('낱말 수 :', len(sample.split()))
print('토큰 수 :', len(pieces))
print('쪼개진 모양:', pieces)
print('\n한 낱말이 여러 조각으로 갈린다 - 그래서 토큰 수가 낱말 수보다 많다')

In [ ]:
# 정말 잘리는지 확인 — 앞부분을 길게 채우고 '맨 뒤'에만 정반대 표현을 붙인다
filler = '선크림 ' * 200                      # 128토큰을 훌쩍 넘긴다
pair = [filler + '정말 최악이에요', filler + '정말 최고예요']
trunc_vecs = model.encode(pair)

score = cosine_similarity(trunc_vecs[0:1], trunc_vecs[1:2])[0][0]
print(f'뒤쪽 문구만 정반대인 두 문장의 유사도: {score:.4f}')
print('1.0 이면 뒷부분이 통째로 잘려 두 문장이 같은 벡터가 된 것이다')

### 🖐️ 함께 따라하기 — 짧게 줄이면 달라질까
위에서는 앞부분이 너무 길어 뒤가 잘렸습니다. **길이를 확 줄이면** 같은 뒷문구가 살아나서 유사도가 떨어지는지 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) short = '선크림 ' * 3 으로 짧은 앞부분을 만든다
# 2) short_pair = [short + '정말 최악이에요', short + '정말 최고예요'] 를 만든다
# 3) model.encode(short_pair) 로 벡터를 만든다
# 4) 두 벡터의 코사인 유사도를 소수 넷째 자리까지 출력한다
# 5) 위의 1.0 과 비교해 본다 — 잘리지 않으면 '최악 vs 최고'가 구분된다

### ✅ 바로 확인 퀴즈

**1.** 긴 문서를 임베딩했더니 서로 다른 문서인데 유사도가 1.00 에 가깝게 나옵니다. 가장 먼저 의심할 것은 무엇인가요?

<details><summary>정답 보기</summary>

**`max_seq_length` 초과로 뒷부분이 잘렸을 가능성**입니다. 앞부분이 같은 문서들이라면 잘린 뒤 남은 내용이 거의 같아져 유사도가 1 에 가까워집니다. 모델의 최대 입력 길이를 확인하고, 길이가 부족하면 더 긴 입력을 받는 모델을 쓰거나 문서를 나눠서 임베딩해야 합니다.

</details>

**2.** 회사 서비스에 넣을 임베딩 모델을 고를 때, 성능 점수 말고 **반드시** 확인해야 하는 것은?

<details><summary>정답 보기</summary>

**라이선스**입니다. 상업적 이용이 금지된 모델(예: 비영리 전용)을 서비스에 쓰면 문제가 됩니다. 함께 **한국어 지원**과 **최대 입력 길이**도 우리 데이터에 맞는지 확인해야 합니다.

</details>

**3.** 벤치마크 순위가 가장 높은 모델을 고르면 항상 최선일까요?

<details><summary>정답 보기</summary>

아닙니다. 벤치마크는 **남의 데이터**로 잰 점수입니다. 우리 데이터의 말투·길이·분야가 다르면 순위가 뒤집히기도 합니다. 후보를 2~3개로 좁힌 뒤 **내 데이터로 직접 비교해 보는 것**이 가장 확실합니다.

</details>

---
# 7. 임베딩 시각화 — 768차원을 2차원 그림으로

## 왜 필요할까요?
임베딩은 768차원이라 **사람이 눈으로 볼 수 없습니다**. "비슷한 문장은 가깝다"는 말을 **그림으로 확인**하려면 차원을 2개로 줄여 평면에 찍어야 합니다. 이렇게 차원을 줄이는 일을 **차원축소(dimensionality reduction)** 라 하고, 우리는 **UMAP** 하나를 씁니다.

### UMAP 의 아이디어 — 이웃 관계를 지킨다
**UMAP**(Uniform Manifold Approximation and Projection)은 각 점의 **가까운 이웃이 누구인지**를 먼저 정리한 뒤, **그 이웃 관계가 최대한 유지되도록** 점들을 2차원 평면에 다시 배치합니다. 그래서 원래 공간에서 한 무리였던 문장들이 그림에서도 **한 덩어리**로 모입니다. 빠르고, 덩어리가 눈에 잘 보여 임베딩 시각화의 사실상 표준 도구입니다.

### 문법
- **`UMAP(n_components=2, n_neighbors=5, min_dist=0.05, random_state=0).fit_transform(임베딩)`** → `(문장 수, 2)` 좌표.
- **`n_components=2`** — 몇 차원으로 줄일지(그림이므로 2).
- **`n_neighbors`** — **각 점의 이웃을 몇 명까지 볼지**. 작게 주면 작은 덩어리가 또렷해지고, 크게 주면 전체 모양을 함께 봅니다. 리뷰가 15건뿐이라 **작게(5)** 둡니다.
- **`min_dist`** — 그림에서 점들이 얼마나 붙어도 되는지(작을수록 덩어리가 촘촘).
- **`random_state`** — 고정해야 **매번 같은 그림**이 나옵니다.

> 아래 그림에서 **같은 장르 리뷰(같은 색)** 끼리 뭉쳐 있으면, 임베딩이 장르 차이를 좌표로 잘 옮겼다는 뜻입니다.

<img src="images/교안_umap_scatter.png" width="640">

In [ ]:
# UMAP 으로 768차원 임베딩을 2차원으로 줄여 장르별 색으로 흩뿌린다
umap_coords = UMAP(n_components=2, n_neighbors=5, min_dist=0.05,
                   random_state=0).fit_transform(embeddings)
print('UMAP 좌표 모양:', umap_coords.shape, '  (리뷰 15건 × 2차원)')

plt.figure(figsize=(7, 5.5))
for genre, color in GENRE_COLORS.items():
    mask = genres == genre
    plt.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                s=90, color=color, alpha=0.85, label=genre)
plt.title('영화 리뷰 임베딩의 2차원 지도 (UMAP)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.show()
print('같은 장르(같은 색) 리뷰끼리 모여 있다 — 임베딩이 장르를 좌표로 잘 옮겼다')

## ⚠️ 그림은 '요약'이지 정답 지도가 아닙니다

같은 임베딩이라도 **`n_neighbors` 를 바꾸면 그림이 달라집니다**. 이웃을 적게 보면 작은 덩어리가 또렷하게 갈라지고, 많이 보면 덩어리가 느슨하게 풀립니다. 아래 두 장을 나란히 그려 확인해 보세요.

여기서 얻을 교훈은 두 가지입니다.

1. **UMAP 은 덩어리를 실제보다 또렷하게 보이도록 그립니다.** 그림이 깔끔하다고 해서 원래 768차원 공간에서도 그만큼 깔끔히 갈린 것은 아닙니다.
2. 그래서 **판단(유사도·검색)은 원래 임베딩에서 잰 값**으로 하고, 2차원 그림은 **눈으로 감을 잡을 때** 씁니다.

In [ ]:
# 같은 임베딩을 n_neighbors 만 바꿔 두 장으로 그려 비교한다
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, nn in zip(axes, [2, 12]):
    alt = UMAP(n_components=2, n_neighbors=nn, min_dist=0.05,
               random_state=0).fit_transform(embeddings)
    for genre, color in GENRE_COLORS.items():
        mask = genres == genre
        ax.scatter(alt[mask, 0], alt[mask, 1], s=90, color=color, alpha=0.85, label=genre)
    ax.set_title(f'UMAP n_neighbors={nn}')
    ax.legend()
plt.tight_layout()
plt.show()
print('같은 데이터·같은 도구인데 그림 모양이 달라진다 — 2차원 그림은 하나의 요약일 뿐이다')

### 🖐️ 함께 따라하기 — UMAP 좌표를 표로 보기

UMAP 으로 줄인 2차원 좌표를 장르와 함께 표로 정리해, 어떤 리뷰가 어디에 찍혔는지 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) coords2d = UMAP(n_components=2, n_neighbors=5, min_dist=0.05,
#                    random_state=0).fit_transform(embeddings) 로 2차원 좌표를 구한다
# 2) coord_df = pd.DataFrame(coords2d, columns=['x', 'y']) 를 만든다
# 3) coord_df['genre'] = df['genre'] 로 장르 열을 붙인다
# 4) coord_df 를 display 해 좌표와 장르를 함께 확인한다

### ✅ 바로 확인 퀴즈

**1.** 왜 임베딩을 **바로** 그리지 못하고 UMAP 으로 줄여서 그릴까요?

<details><summary>정답 보기</summary>

임베딩은 **768차원**이라 사람이 눈으로 볼 수 없기 때문입니다. UMAP 이 **이웃 관계를 최대한 유지하며** 2차원으로 펼쳐 줘야 산점도로 확인할 수 있습니다.

</details>

**2.** 2차원 그림에서 **같은 장르 리뷰끼리 뭉쳐** 있으면 무엇을 알 수 있나요?

<details><summary>정답 보기</summary>

임베딩이 **장르(뜻)의 차이를 좌표로 잘 담았다**는 뜻입니다. 뜻이 비슷한 같은 장르 문장이 가까운 벡터가 되었기에, 2차원으로 줄여도 같은 색끼리 모입니다.

</details>

---
## 이번 강의 정리

오늘은 문장의 **의미를 좌표로** 옮기는 임베딩을 배우고, 유사도·검색·시각화까지 이었습니다.

| 주제 | 핵심 | 함수 |
|---|---|---|
| 희소 vs 밀집 | 단어 세기(0 많음) → 의미 좌표(실수) | (개념) |
| 문장 → 벡터 | BERT 토큰 벡터 → 풀링 → 768차원 문장 임베딩(SBERT) | `model.encode` |
| 유사도 | 방향으로 뜻 비교(−1~1) | `cosine_similarity` |
| 다른 척도 | 유클리드는 **작을수록**·내적은 길이까지; 정규화하면 셋이 일치 | `euclidean_distances` · `@` |
| 의미 검색 | 질문 임베딩 → top-k | `np.argsort(-scores)` |
| 상용 API | 결과는 똑같이 벡터(1536차원) | `client.embeddings.create` |
| 모델 고르기 | 언어·차원·**최대 입력 길이**·라이선스 | 허브 카드 · `model.max_seq_length` |
| 시각화 | 768차원 → 2차원 그림 | `UMAP` |

- **임베딩**은 단어가 겹치지 않아도 **뜻이 가까운 문장**을 찾게 해 줍니다(TF-IDF 의 한계 극복).
- **코사인 유사도**로 두 벡터의 방향을 비교하고, 그걸 쌓으면 **의미 검색**이 됩니다.
- 재는 법은 셋(**코사인·유클리드 거리·내적**)이고 **정렬 방향이 서로 다릅니다** — L2 정규화해 두면 셋의 검색 결과가 같아집니다.
- HuggingFace(로컬·무료)든 상용 API 든 **결과는 벡터**이고 이후 처리는 동일합니다.
- **UMAP** 으로 임베딩을 2차원에 펼쳐 "같은 장르는 가깝다"를 눈으로 확인했습니다. 다만 그림은 **이웃 관계를 살려 펼친 요약**이라, 판단은 원래 768차원에서 잰 값으로 해야 합니다.
- 모델은 **허브에서 직접 고를 수 있습니다** — 특히 `max_seq_length` 를 넘긴 글은 경고 없이 잘리니 긴 문서를 다룰 땐 반드시 확인하세요.

## ⏭️ 예고 — 다음 시간: 임베딩으로 문서를 자동으로 묶기

이번 시간엔 문장을 벡터로 바꾸고 **하나하나 비교**했습니다. 다음 시간엔 그 임베딩을 **한 단계 더** 활용합니다.
- **문서 군집·토픽** — 라벨 없이도 비슷한 문서끼리 **저절로 묶어** 주제를 뽑아냅니다.
- **이미지 임베딩** — 글뿐 아니라 **그림도 벡터**로 바꿔, 텍스트와 이미지를 함께 다룹니다.

오늘 익힌 임베딩과 코사인 유사도가 그 모든 것의 **바탕**이 됩니다. 수고하셨습니다!